# Titanic Passenger Survival Analysis
Our target is to Clean, explore, and analyze the Titanic dataset to uncover key demographic and socio-economic factors that determined passenger survival

In [183]:
import pandas as pd
import seaborn as sns

In [184]:
df = sns.load_dataset('titanic')

### Data Dictionary
* **survived:** Survival status (0 = No, 1 = Yes)
* **pclass:** Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd)
* **sex:** Gender of the passenger
* **age:** Age in years
* **sibsp:** Number of siblings / spouses aboard the Titanic
* **parch:** Number of parents / children aboard the Titanic
* **fare:** Passenger fare paid
* **embarked:** Port of Embarkation (C = Cherbourg, Q = Queenstown, S = Southampton)
* **class:** Ticket class name (First, Second, Third)
* **who:** man, woman, or child
* **adult_male:** Boolean indicator for adult males
* **deck:** Cabin deck level
* **embark_town:** Port of embarkation full name
* **alive:** Survival status as text ('yes' or 'no')
* **alone:** Boolean indicator if passenger was traveling alone

In [185]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [186]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [187]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [188]:
df.duplicated().sum()

np.int64(107)

In [189]:
df.drop_duplicates(inplace=True)

df.duplicated().sum()

np.int64(0)

In [190]:
df.isnull().sum()

survived         0
pclass           0
sex              0
age            106
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           582
embark_town      2
alive            0
alone            0
dtype: int64

In [191]:
# Too many missing values comparing with total number of rows (better to drop the column)
df.drop(columns=['deck'], inplace=True)

# just two missing values each (filling it with the most frequent port)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])

In [192]:
# Group by Passenger Class to see average and median ages
age_insights = df.groupby('pclass')['age'].agg(['mean', 'median', 'count'])
print(age_insights)

             mean  median  count
pclass                          
1       38.310378    37.0    185
2       29.898302    29.0    159
3       25.180150    24.0    334


As the passenger class level gets lower (from 1 to 3), the average age drops significantly, proving that socio-economic status is a brilliant feature to use for conditional age imputation.

## Mean vs Median for Age Imputation

When deciding how to fill missing values in the `age` feature, we must evaluate whether to use the **Mean** or the **Median** of each group

---

### Scenario A: Imputing with the **Mean**
* **Pros:** 
  * Accounts for every single data point proportionally.
  * Works exceptionally well if the age distribution within a class is perfectly symmetric (normal distribution).
* **Cons:** 
  * **Sensitive to Outliers:** Extremely young infants like (e.g., 0.42 years) or elderly passengers (e.g., 80 years) skew the average up or down.
  * **Non-Integer Values:** Produces floating-point averages (e.g., `38.233441` years), which can look unnatural when representing age without further rounding.

---

### Scenario B: Imputing with the **Median**
* **Pros:** 
  * **Robust to Skewness & Outliers:** Extreme values on either tail do not distort the central tendency.
  * **Natural Values:** Produces clean representative numbers (e.g., `37.0` or `24.0` years) that align with real-world demographic records.
* **Cons:** 
  * Ignores the actual values of extreme data points, only considering their positional rank.
  * If a group has a bimodal distribution (e.g., two clusters at 20 and 45), the median can fall into the middle gap where few actual data points exist.

To evaluate both methods on our dataset, let's compare the group statistics across passenger classes (`pclass`):

| Passenger Class (`pclass`) | Group Mean Age | Group Median Age | Difference |
| :--- | :--- | :--- | :--- |
| **1st Class** | ~38.23 years | 37.0 years | ~1.23 years |
| **2nd Class** | ~29.88 years | 29.0 years | ~0.88 years |
| **3rd Class** | ~25.14 years | 24.0 years | ~1.14 years |

By looking at both the mean and the median across passenger classes. Because ages can have outliers, we chose the median to keep our imputed values robust, even though both values are very close.

In [193]:
median_1 = df[df['pclass'] == 1]['age'].median()
median_2 = df[df['pclass'] == 2]['age'].median()
median_3 = df[df['pclass'] == 3]['age'].median()

df.loc[(df['pclass'] == 1) & (df['age'].isnull()), 'age'] = median_1
df.loc[(df['pclass'] == 2) & (df['age'].isnull()), 'age'] = median_2
df.loc[(df['pclass'] == 3) & (df['age'].isnull()), 'age'] = median_3

df.isnull().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [194]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
885,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,Queenstown,no,False
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,Southampton,yes,True
888,0,3,female,24.0,1,2,23.4500,S,Third,woman,False,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,Cherbourg,yes,True


In [195]:
# Check unique values and data types for every column in the dataframe (continuous variables are excluded)
for col in df.columns:
    if col == 'age' or col == 'fare':
        continue
    unique_count = df[col].nunique()
    print(f"Column: '{col}' | Values: {df[col].unique()}")

Column: 'survived' | Values: [0 1]
Column: 'pclass' | Values: [3 1 2]
Column: 'sex' | Values: <StringArray>
['male', 'female']
Length: 2, dtype: str
Column: 'sibsp' | Values: [1 0 3 4 2 5 8]
Column: 'parch' | Values: [0 1 2 5 3 4 6]
Column: 'embarked' | Values: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str
Column: 'class' | Values: ['Third', 'First', 'Second']
Categories (3, str): ['First', 'Second', 'Third']
Column: 'who' | Values: <StringArray>
['man', 'woman', 'child']
Length: 3, dtype: str
Column: 'adult_male' | Values: [ True False]
Column: 'embark_town' | Values: <StringArray>
['Southampton', 'Cherbourg', 'Queenstown']
Length: 3, dtype: str
Column: 'alive' | Values: <StringArray>
['no', 'yes']
Length: 2, dtype: str
Column: 'alone' | Values: [False  True]


By dropping the text columns (class, embarked, alive) and keeping the clean, concise coded versions (pclass, embark_town, survived), our dataset will be ready for both Exploratory Data Analysis (EDA) and Machine Learning.

In [196]:
columns_to_drop = ['class', 'embarked', 'alive']

df.drop(columns=columns_to_drop, inplace=True)

Because we can easily figure out who is an adult male just by looking at 'who', keeping 'adult_male' just adds unnecessary clutter to our dataset

In [197]:
df.drop(columns=['adult_male'], inplace=True)

Converting boolean columns (such as `alone`) from `True`/`False` into binary integers (`1`/`0`) to streamline mathematical and statistical analysis

In [198]:
boolean_cols = ['alone']

for col in boolean_cols:
    df[col] = df[col].astype(int)

In [199]:
df

,survived,pclass,sex,age,sibsp,parch,fare,who,embark_town,alone
0,0,3,male,22.0,1,0,7.2500,man,Southampton,0
1,1,1,female,38.0,1,0,71.2833,woman,Cherbourg,0
2,1,3,female,26.0,0,0,7.9250,woman,Southampton,1
3,1,1,female,35.0,1,0,53.1000,woman,Southampton,0
4,0,3,male,35.0,0,0,8.0500,man,Southampton,1
...,...,...,...,...,...,...,...,...,...,...
885,0,3,female,39.0,0,5,29.1250,woman,Queenstown,0
887,1,1,female,19.0,0,0,30.0000,woman,Southampton,1
888,0,3,female,24.0,1,2,23.4500,woman,Southampton,0
889,1,1,male,26.0,0,0,30.0000,man,Cherbourg,1


A passenger is only truly "alone" if they have 0 siblings/spouses (sibsp == 0) and 0 parents/children (parch == 0). If alone says 1 (True), but they actually have family members on board, that's a contradiction in the data

In [200]:
# Find rows where 'alone' is 1 (True), but sibsp or parch is greater than 0
inconsistent_rows = df[(df['alone'] == 1) & ((df['sibsp'] > 0) | (df['parch'] > 0))]
print(inconsistent_rows)

Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, who, embark_town, alone]
Index: []


In [201]:
# Find rows where someone has no family (sibsp=0 and parch=0), but 'alone' is 0 (False)
reverse_inconsistent = df[(df['alone'] == 0) & (df['sibsp'] == 0) & (df['parch'] == 0)]
print(reverse_inconsistent)

Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, who, embark_town, alone]
Index: []


In [202]:
# Export the cleaned dataset to a CSV file
df.to_csv('titanic_cleaned.csv', index=False)